In [ ]:
!pip install detoxify gradio matplotlib torch -q

In [ ]:
# Alternative: use sample data (no Kaggle needed)
import pandas as pd

data = {
    'user_id': ['user_A']*6 + ['user_B']*6 + ['user_C']*6,
    'comment': [
        # User A - escalating (DANGER)
        "I really enjoy watching these videos",
        "Some of these comments are a bit annoying",
        "Why do people even post such garbage here",
        "I hate this community so much",
        "You people are complete idiots",
        "Everyone here deserves to suffer",
        # User B - stable safe
        "Great video today!",
        "I learned something new",
        "Thanks for sharing this",
        "Really helpful content",
        "Keep up the good work",
        "Looking forward to more",
        # User C - mild drift (WATCH)
        "Nice content overall",
        "This could be better honestly",
        "Getting a bit repetitive",
        "Not my favorite episode",
        "This is kind of stupid",
        "Pretty bad quality lately"
    ],
    'timestamp': pd.date_range('2024-01-01', periods=18, freq='12H'),
    'post_freq_per_day': [1,1,2,3,8,15, 1,1,1,2,1,1, 1,1,2,2,3,4]
}

df = pd.DataFrame(data)
print(df.head(10))
print(f"\nTotal comments: {len(df)}")
print(f"Users: {df['user_id'].unique()}")

  user_id                                    comment           timestamp  \
0  user_A       I really enjoy watching these videos 2024-01-01 00:00:00   
1  user_A  Some of these comments are a bit annoying 2024-01-01 12:00:00   
2  user_A  Why do people even post such garbage here 2024-01-02 00:00:00   
3  user_A              I hate this community so much 2024-01-02 12:00:00   
4  user_A             You people are complete idiots 2024-01-03 00:00:00   
5  user_A           Everyone here deserves to suffer 2024-01-03 12:00:00   
6  user_B                         Great video today! 2024-01-04 00:00:00   
7  user_B                    I learned something new 2024-01-04 12:00:00   
8  user_B                    Thanks for sharing this 2024-01-05 00:00:00   
9  user_B                     Really helpful content 2024-01-05 12:00:00   

   post_freq_per_day  
0                  1  
1                  1  
2                  2  
3                  3  
4                  8  
5                 15  
6 

/tmp/ipykernel_26988/4010702684.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'timestamp': pd.date_range('2024-01-01', periods=18, freq='12H'),


In [ ]:

from detoxify import Detoxify
import numpy as np

print("Loading model...")
model = Detoxify('original')

scores = []
for comment in df['comment']:
    result = model.predict(comment)
    scores.append(round(result['toxicity'], 3))

df['toxicity_score'] = scores
print("Done! Toxicity scores added.")
print(df[['user_id', 'comment', 'toxicity_score']])

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done! Toxicity scores added.
   user_id                                    comment  toxicity_score
0   user_A       I really enjoy watching these videos           0.001
1   user_A  Some of these comments are a bit annoying           0.004
2   user_A  Why do people even post such garbage here           0.827
3   user_A              I hate this community so much           0.390
4   user_A             You people are complete idiots           0.985
5   user_A           Everyone here deserves to suffer           0.921
6   user_B                         Great video today!           0.001
7   user_B                    I learned something new           0.001
8   user_B                    Thanks for sharing this           0.001
9   user_B                     Really helpful content           0.001
10  user_B                      Keep up the good work           0.001
11  user_B                    Looking forward to more           0.001
12  user_C                       Nice content overall        

In [ ]:
def analyze_user(user_df):
    scores = user_df['toxicity_score'].values
    freqs = user_df['post_freq_per_day'].values

    # Split into early vs late
    half = len(scores) // 2
    early_avg = np.mean(scores[:half])
    late_avg = np.mean(scores[half:])
    drift = round(late_avg - early_avg, 3)

    # Frequency spike
    freq_spike = round(freqs[-1] / freqs[0], 1) if freqs[0] > 0 else 1.0

    # Combined risk score
    combined = drift + (0.2 if freq_spike > 3 else 0)
    avg_tox = round(np.mean(scores), 3)

    # Risk level
    if combined > 0.45 or avg_tox > 0.65:
        risk = "DANGER"
    elif combined > 0.2 or avg_tox > 0.4:
        risk = "WATCH"
    elif drift < -0.2:
        risk = "IMPROVING"
    else:
        risk = "SAFE"

    return {
        'avg_toxicity': avg_tox,
        'drift_score': drift,
        'freq_spike': f"{freq_spike}x",
        'risk_level': risk,
        'scores': scores.tolist(),
        'comments': user_df['comment'].tolist()
    }

# Analyze all users
results = {}
for user in df['user_id'].unique():
    user_df = df[df['user_id'] == user].reset_index(drop=True)
    results[user] = analyze_user(user_df)
    print(f"\n{user}: {results[user]['risk_level']} | drift={results[user]['drift_score']} | freq={results[user]['freq_spike']}")


user_A: DANGER | drift=0.4880000054836273 | freq=15.0x

user_B: SAFE | drift=0.0 | freq=1.0x

user_C: DANGER | drift=0.3070000112056732 | freq=4.0x


In [ ]:

try:
    from detoxify import Detoxify
except:
    import subprocess
    subprocess.run(["pip", "install", "detoxify", "-q"])
    from detoxify import Detoxify

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
# ... rest of your code
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
from detoxify import Detoxify

model = Detoxify('original')

TOXIC_WORDS = ['hate','kill','idiot','stupid','garbage',
               'worthless','disgusting','moron','loser',
               'dumb','pathetic','awful','terrible','destroy','suffer']

def highlight_toxic_words(comment):
    for word in TOXIC_WORDS:
        comment = comment.replace(word, f'[**{word}**]')
    return comment

def full_analysis(comments_text, freq_text):
    comments = [c.strip() for c in comments_text.strip().split('\n') if c.strip()]
    if len(comments) < 2:
        return "Please enter at least 2 comments.", None, "Unknown", "No data"

    # Score comments
    scores = []
    for c in comments:
        result = model.predict(c)
        scores.append(round(result['toxicity'], 3))

    # Parse frequencies
    freqs = []
    if freq_text.strip():
        try:
            freqs = [float(x.strip()) for x in freq_text.split(',')]
        except:
            freqs = []

    # Drift calculation
    half = len(scores) // 2
    early_avg = np.mean(scores[:half])
    late_avg = np.mean(scores[half:])
    drift = round(late_avg - early_avg, 3)
    avg_tox = round(np.mean(scores), 3)

    # Frequency spike
    freq_spike = 1.0
    if len(freqs) >= 2:
        freq_spike = round(freqs[-1] / freqs[0], 1)

    # Risk level
    combined = drift + (0.2 if freq_spike > 3 else 0)
    if combined > 0.45 or avg_tox > 0.65:
        risk = "DANGER"
    elif combined > 0.2 or avg_tox > 0.4:
        risk = "WATCH"
    elif drift < -0.2:
        risk = "IMPROVING"
    else:
        risk = "SAFE"

    # Chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    risk_colors = {'DANGER':'#E24B4A','WATCH':'#EF9F27','SAFE':'#639922','IMPROVING':'#378ADD'}
    col = risk_colors[risk]

    # Toxicity timeline
    ax1.plot(range(1, len(scores)+1), scores,
             marker='o', color=col, linewidth=2.5,
             markersize=8, markerfacecolor='white',
             markeredgewidth=2.5, markeredgecolor=col)
    ax1.fill_between(range(1, len(scores)+1), scores, alpha=0.1, color=col)
    ax1.axhline(y=0.5, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Warning (0.5)')
    ax1.axhline(y=0.7, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Danger (0.7)')
    ax1.set_ylim(0, 1)
    ax1.set_title(f'Toxicity Timeline — {risk}', fontweight='bold', color=col, fontsize=13)
    ax1.set_xlabel('Comment Number')
    ax1.set_ylabel('Toxicity Score')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.text(0.98, 0.98, risk,
             transform=ax1.transAxes,
             fontsize=11, fontweight='bold', color='white',
             bbox=dict(boxstyle='round,pad=0.3', facecolor=col, alpha=0.9),
             ha='right', va='top')

    # Frequency chart
    if freqs:
        bar_colors = ['#E24B4A' if f > freqs[0]*3 else '#378ADD' for f in freqs]
        ax2.bar(range(1, len(freqs)+1), freqs, color=bar_colors, alpha=0.85, width=0.6)
        ax2.set_title('Posting Frequency Pattern', fontweight='bold', fontsize=13)
        ax2.set_xlabel('Day')
        ax2.set_ylabel('Posts per Day')
        ax2.grid(True, alpha=0.3, axis='y')
        for i, f in enumerate(freqs):
            ax2.text(i+1, f+0.3, str(int(f)), ha='center', fontsize=10, fontweight='bold')
    else:
        ax2.text(0.5, 0.5, 'No frequency\ndata provided',
                 ha='center', va='center', fontsize=13,
                 transform=ax2.transAxes, color='gray')
        ax2.set_title('Posting Frequency Pattern', fontweight='bold', fontsize=13)

    plt.tight_layout()

    # Text output — FIXED rounding
    risk_emoji = {'DANGER':'🚨','WATCH':'⚠️','SAFE':'✅','IMPROVING':'📈'}
    output = f"{risk_emoji[risk]} RISK LEVEL: {risk}\n"
    output += f"{'='*40}\n"
    output += f"Average Toxicity : {round(avg_tox, 3)}\n"
    output += f"Drift Score      : {drift:+.3f}\n"
    output += f"Frequency Spike  : {freq_spike}x\n\n"
    output += "COMMENT BREAKDOWN:\n"
    output += f"{'-'*40}\n"
    for i, (c, s) in enumerate(zip(comments, scores)):
        level = "HIGH" if s > 0.6 else "MID" if s > 0.35 else "LOW"
        output += f"[{i+1}] Score: {round(s, 3)} ({level})\n    {highlight_toxic_words(c)}\n\n"

    if risk == 'DANGER':
        output += "INTERVENTION RECOMMENDED:\n"
        if drift > 0.4:
            output += "- Strong toxicity escalation detected\n"
        if freq_spike > 3:
            output += f"- Posting frequency spiked {freq_spike}x\n"
        output += "- Flag user for moderator review\n"

    return output, fig, risk, f"Drift: {drift:+.3f} | Avg Toxicity: {avg_tox} | Freq Spike: {freq_spike}x"


# Gradio UI
with gr.Blocks(title="Behavioral Drift Detector", theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🧠 Behavioral Drift & Toxicity Escalation Detector
    ### AI-powered early warning system for online platforms
    > Detects gradual behavioral shifts **before** they become harmful
    """)

    with gr.Row():
        with gr.Column(scale=1):
            comments_input = gr.Textbox(
                lines=8,
                label="💬 User Comments (one per line, oldest first)",
                placeholder="i love this content\nthis is getting annoying\ni dislike this community\ni hate everyone here\nyou are all idiots\nyou deserve to suffer"
            )
            freq_input = gr.Textbox(
                lines=1,
                label="📊 Posting Frequency (posts/day, comma separated)",
                placeholder="1, 2, 4, 8, 15, 20"
            )
            analyze_btn = gr.Button("🔍 Analyze Behavior", variant="primary")

        with gr.Column(scale=1):
            risk_output = gr.Textbox(label="🚦 Risk Level", lines=1)
            stats_output = gr.Textbox(label="📈 Key Stats", lines=1)
            text_output = gr.Textbox(label="📋 Full Analysis", lines=12)

    chart_output = gr.Plot(label="📊 Visual Analysis")

    analyze_btn.click(
        fn=full_analysis,
        inputs=[comments_input, freq_input],
        outputs=[text_output, chart_output, risk_output, stats_output]
    )

    gr.Markdown("### 📝 Try these examples")
    gr.Examples(
        examples=[
            ["i love this content\nthis is getting annoying\ni dislike this community\ni hate everyone here\nyou are all idiots\nyou deserve to suffer", "1, 2, 4, 8, 15, 20"],
            ["great video!\nreally helpful\nthanks for sharing\nkeep it up\namazing work\nlove this channel", "1, 1, 1, 1, 2, 1"],
            ["nice video\ngetting repetitive\nthis is kind of stupid\npretty bad quality", "1, 1, 2, 3"]
        ],
        inputs=[comments_input, freq_input]
    )

app.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_26988/565890596.py:138: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Behavioral Drift Detector", theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e81f9bcbdf0b15e2b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
